# NeuroVascNet — EEG → hemodynamics, complete pipeline

**Self-contained.** Downloads its own data, builds every cache from raw, and runs the full
cross-validated evaluation. Nothing needs to be uploaded or pre-computed.

## Purpose

One EEG encoder that, at deployment, takes **EEG only** and predicts the haemodynamic
response — fNIRS (HbO/HbR) and fMRI (BOLD). Training uses paired recordings; deployment
does not.

## What this pipeline does

1. Downloads TU Berlin hBCI and OpenNeuro ds002336 from Google Drive
2. Converts raw 760/850 nm fNIRS intensities to HbO/HbR with its own Beer–Lambert
3. Rebuilds the fMRI target: event-locked via R128 scanner triggers, motion + drift
   regressed, restricted to sensorimotor ROIs
4. Gates every target on **split-half reliability** before believing any prediction
5. Evaluates with **subject-wise 5-fold cross-validation** — every subject held out once
6. Scores against a **constant-predictor baseline** and a 1000-fold permutation null

## What we achieved

| | result |
|---|---|
| fNIRS target repaired | reliability **0.207 → 0.440**, HbO/HbR sign corrected |
| fMRI target repaired | reliability **−0.639 → +0.638** |
| A false positive caught | broken target gave r=0.0706 **p=0.005**; repaired gives **p=0.473** |
| Real cross-subject EEG signal | subject-demeaned **r ≈ 0.053–0.071, p = 0.005** |

## What we did NOT achieve

* **The effect is small** — roughly 11% of the achievable ceiling.
* **The raw metric is misleading.** Plain correlation gives 0.072, but a predictor that
  *ignores EEG entirely* scores 0.076 — it wins. Trial averaging widens that gap
  (0.243 vs 0.309). A bigger raw number is a **worse** result. Judge on `r_demeaned`.
* **Amplitude prediction failed** on the repaired data (p ≥ 0.07 everywhere).
* **fMRI has no positive result** — underpowered rather than disproven (263 trials vs 1380).
* Reliability bounds what is *achievable*, not what is *reachable*: a response can be
  perfectly reproducible while its trial-to-trial variation is driven by vascular and
  systemic factors EEG cannot see.

## Setup

* **GPU** strongly recommended (Kaggle: Settings → Accelerator → GPU)
* **Internet on** (Kaggle: Settings → Internet → On)
* Run all cells.

| stage | time |
|---|---|
| download + build fNIRS (6.6 GB) | 20–60 min |
| download + build fMRI (18 GB) | 40–90 min |
| cross-validated evaluation + deep training | ~45 min |

In [ ]:
import os, sys, shutil, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd

ON_KAGGLE = Path("/kaggle").exists()
WORK  = Path("/kaggle/working" if ON_KAGGLE else ".")
RAW   = WORK / "raw";   RAW.mkdir(parents=True, exist_ok=True)
CACHE = WORK / "cache"; CACHE.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------- config
BUILD_COMPARISON = False   # True also downloads HEFMI (13 GB) to reproduce the
                           # before/after that demonstrates the false positive
FREE_RAW_AFTER   = True    # delete each dataset's raw files once its cache exists
                           # keep True on Kaggle: /kaggle/working is capped at ~20 GB

K_FOLDS   = 5
N_PERM    = 1000
N_BOOT    = 2000
MAX_EPOCHS, PATIENCE, BS = 400, 40, 32
LR_GRID   = (3e-4, 1e-3, 3e-3)

EEG_FS    = 200.0
BANDS     = [(1,4),(4,8),(8,13),(13,30),(30,45)]
N_BINS    = 6
RESP_WIN  = (4.0, 14.0)

DRIVE = {
    "hbci":     "1lw-AR79e5FgVOL6iHaRG0FppYJHOhnqj",   # TU Berlin hBCI     6.6 GB
    "ds002336": "1K_91Y5Ph0PiAIcFdlGsiHg10_knUNmhw",   # OpenNeuro ds002336  18 GB
    "hefmi":    "1OnC-Z62tLH8ompyxj4ev_qABiqIkj8jv",   # HEFMI (comparison)  13 GB
}
DIRNAME = {"hbci": "TU Berlin hBCI EEG+FNRI", "ds002336": "OpenNeuro ds002336",
           "hefmi": "HEFMI-ICH EEG+FNRI"}

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "gdown", "mne", "nilearn", "nibabel"], check=False)

try:
    import torch
    DEV = "cuda" if torch.cuda.is_available() else "cpu"
    print("torch", torch.__version__, "| device:", DEV,
          "|", torch.cuda.get_device_name(0) if DEV=="cuda" else "CPU (slow)")
except Exception as e:
    DEV = "cpu"; print("torch unavailable:", e)
print("workdir:", WORK)

## 1. Download helper

`gdown.download_folder` silently caps at 50 files per folder unless `remaining_ok=True` —
these folders hold hundreds, so that flag is essential. Google Drive rate-limits multi-GB
pulls; if it stops early, just re-run the cell. Finished files are skipped, so it resumes.

In [ ]:
def fetch(key):
    dest = RAW / DIRNAME[key]
    if dest.exists() and len(list(dest.rglob("*"))) > 10:
        print(f"{DIRNAME[key]}: already present ({len(list(dest.rglob('*')))} files)")
        return dest
    import gdown
    print(f"downloading {DIRNAME[key]} ...")
    try:
        gdown.download_folder(id=DRIVE[key], output=str(dest), quiet=False,
                              use_cookies=False, remaining_ok=True)
    except Exception as e:
        print(f"  interrupted: {type(e).__name__}: {e}")
        print("  re-run this cell to resume")
    print(f"  {len(list(dest.rglob('*')))} files")
    return dest

def free(key):
    if not FREE_RAW_AFTER: return
    d = RAW / DIRNAME[key]
    if d.exists():
        shutil.rmtree(d, ignore_errors=True)
        print(f"  freed {DIRNAME[key]}")

def cached(*names):
    return all((CACHE/n).exists() for n in names)

## 2. fNIRS arm — TU Berlin hBCI, built from raw wavelengths

The dataset ships **raw 760/850 nm intensities** (36 source–detector pairs × 2 wavelengths),
so the Beer–Lambert conversion is done here with published extinction coefficients rather
than trusting a vendor pipeline. That matters: the dataset this replaced had HbO/HbR =
**+0.898**, physiologically impossible for a real response and the signature of a bad
conversion.

Global-signal regression is deliberately **not** applied — it removes ~83% of the evoked
response, because a broad motor response *is* spatially global.

In [ ]:
import scipy.io as sio
from scipy import signal as sps

EXT = {760: {"HbO":1486.59, "HbR":3843.71}, 850: {"HbO":2526.39, "HbR":1798.64}}
DPF = {760: 6.4, 850: 5.8}
SD_CM, EOG = 3.0, ("VEOG","HEOG")
TMIN, TMAX, EEG_SAMPLES, HEMO_SAMPLES = -2.0, 28.0, 6000, 300

def _runs(p, hint=None):
    m = sio.loadmat(p, squeeze_me=True, struct_as_record=False)
    ks = [k for k in m if not k.startswith("__")]
    return np.atleast_1d(m[hint if hint in ks else ks[0]]).ravel()

def _bp(x, fs, lo, hi, order=3):
    nyq = fs/2; hi = min(hi, nyq*0.95)
    if lo >= hi: return x
    b, a = sps.butter(order, [lo/nyq, hi/nyq], btype="band")
    return sps.filtfilt(b, a, x, axis=-1)

def beer_lambert(lo_wl, hi_wl):
    def od(x):
        base = np.median(x, axis=-1, keepdims=True)
        base = np.where(np.abs(base) < 1e-12, 1e-12, base)
        return -np.log(np.clip(x/base, 1e-9, None))
    a1, a2 = od(lo_wl), od(hi_wl)
    L1, L2 = SD_CM*DPF[760], SD_CM*DPF[850]
    E = np.array([[EXT[760]["HbO"]*L1, EXT[760]["HbR"]*L1],
                  [EXT[850]["HbO"]*L2, EXT[850]["HbR"]*L2]])
    Ei = np.linalg.inv(E)
    return (Ei[0,0]*a1 + Ei[0,1]*a2)*1e6, (Ei[1,0]*a1 + Ei[1,1]*a2)*1e6

def _split_wl(x, clab):
    names = [str(c) for c in np.atleast_1d(clab)]
    lo = {n.replace("lowWL",""):i for i,n in enumerate(names) if "lowWL" in n}
    hi = {n.replace("highWL",""):i for i,n in enumerate(names) if "highWL" in n}
    pr = sorted(set(lo) & set(hi))
    return np.stack([x[:,lo[p]] for p in pr]), np.stack([x[:,hi[p]] for p in pr])

def _rs(x, n): return x if x.shape[-1]==n else sps.resample(x, n, axis=-1)

def build_hbci(root, task="MI"):
    out = CACHE/f"hbci_{task}.npz"
    if out.exists(): print(f"hbci {task}: cached"); return out
    eeg_o, hemo_o, sid_o = [], [], []
    nb = int(2.0*10)
    for s in range(1, 30):
        ed = None
        for g in sorted((root/"EEG data").glob("EEG_*")):
            c = g/f"subject {s:02d}"/"with occular artifact"
            if c.exists(): ed = c; break
        nd = root/"NRI data"/"NIRS_01-29"/f"subject {s:02d}"
        if ed is None or not nd.exists(): continue
        try:
            er, em = _runs(ed/"cnt.mat","cnt"), _runs(ed/"mrk.mat","mrk")
            nr, nm = _runs(nd/"cnt.mat","cnt"), _runs(nd/"mrk.mat","mrk")
        except Exception as e:
            print(f"  subj {s:02d}: {e}"); continue
        cnt = 0
        for r in range(min(er.size, nr.size)):
            E, N = er[r], nr[r]
            if not str(getattr(E,"title","")).upper().startswith(task): continue
            efs = float(np.atleast_1d(E.fs).ravel()[0])
            clab = [str(c) for c in np.atleast_1d(E.clab)]
            keep = [i for i,c in enumerate(clab) if c not in EOG]
            ex = np.asarray(E.x, np.float64)[:, keep].T
            nfs = float(np.atleast_1d(N.fs).ravel()[0])
            lo, hi = _split_wl(np.asarray(N.x, np.float64), N.clab)
            hbo, hbr = beer_lambert(lo, hi)
            hbo, hbr = _bp(hbo,nfs,0.01,0.2), _bp(hbr,nfs,0.01,0.2)
            if r >= em.size or r >= nm.size: continue
            eon = np.atleast_1d(np.asarray(em[r].time, float))/1000.0
            non = np.atleast_1d(np.asarray(nm[r].time, float))/1000.0
            for k in range(min(eon.size, non.size)):
                ae,be = int(round((eon[k]+TMIN)*efs)), int(round((eon[k]+TMAX)*efs))
                an,bn = int(round((non[k]+TMIN)*nfs)), int(round((non[k]+TMAX)*nfs))
                if ae<0 or be>ex.shape[1] or an<0 or bn>hbo.shape[1]: continue
                se = _rs(ex[:,ae:be], EEG_SAMPLES)
                se = ((se-se.mean(-1,keepdims=True))/(se.std(-1,keepdims=True)+1e-6)).astype(np.float32)
                so, sr = _rs(hbo[:,an:bn],HEMO_SAMPLES), _rs(hbr[:,an:bn],HEMO_SAMPLES)
                so = so - so[:,:nb].mean(-1,keepdims=True)
                sr = sr - sr[:,:nb].mean(-1,keepdims=True)
                eeg_o.append(se); hemo_o.append(np.concatenate([so,sr],0).astype(np.float32))
                sid_o.append(s); cnt += 1
        print(f"  subj {s:02d}: {cnt} trials")
    np.savez_compressed(out, eeg=np.stack(eeg_o), hemo=np.stack(hemo_o),
                        subject_ids=np.asarray(sid_o))
    print(f"{out.name}: {len(eeg_o)} trials, {len(set(sid_o))} subjects")
    return out

if not cached("hbci_MI.npz"):
    root = fetch("hbci")
    build_hbci(root, "MI")
    free("hbci")
else:
    print("hbci_MI.npz: cached")

## 3. fMRI arm — ds002336, event-locked rebuild

The original target used 30 s sliding windows with no nuisance regression over all 48
Harvard–Oxford ROIs, and had split-half reliability of **−0.793** — it did not reproduce
against itself, so every number computed against it was spurious. Three fixes:

1. **event-locked** on stimulus onsets rather than sliding windows (the decisive one)
2. **motion + polynomial drift regressed out** before epoching
3. **restricted to sensorimotor ROIs**

EEG↔BOLD alignment uses the **R128 scanner triggers**, which mark every volume — so volume
*k* is EEG sample `R128[k]` exactly, with no assumed offset.

In [ ]:
import mne, nibabel as nib
from nilearn.datasets import fetch_atlas_harvard_oxford
from nilearn.image import resample_to_img

TR, EPOCH_VOL, N_ROI = 2.0, 15, 48
MOTOR_KEYS = ("Precentral","Postcentral","Supplementary","Juxtapositional")
FMRI_TASKS = ("motorloc","MIpre","MIpost","eegNF","fmriNF","eegfmriNF")

def _clean(ts, motion):
    n = ts.shape[0]; t = np.linspace(-1,1,n)
    cols = [np.ones(n), t, t**2, t**3]
    if motion is not None and motion.shape[0] >= 2:
        m = motion[:n] if motion.shape[0]>=n else np.vstack(
            [motion, np.repeat(motion[-1:], n-motion.shape[0], 0)])
        for k in range(min(6, m.shape[1])):
            v = m[:,k]; sd = v.std()
            cols.append((v-v.mean())/(sd if sd>0 else 1.0))
    X = np.column_stack(cols)
    return ts - X @ np.linalg.lstsq(X, ts, rcond=None)[0]

def build_fmri(root):
    out = CACHE/"ds002336_fixed.npz"
    if out.exists(): print("fMRI: cached"); return out
    atlas = fetch_atlas_harvard_oxford("cort-maxprob-thr25-2mm")
    amap = atlas.maps if hasattr(atlas,"maps") else atlas["maps"]
    labels = [str(x) for x in (atlas.labels if hasattr(atlas,"labels") else atlas["labels"])[1:N_ROI+1]]
    motor = [i for i,nm in enumerate(labels) if any(k in nm for k in MOTOR_KEYS)]
    print("motor ROIs:", [labels[i] for i in motor])

    eeg_o, hemo_o, sid_o = [], [], []
    for si, sub in enumerate(sorted(d.name for d in root.glob("sub-*") if d.is_dir())):
        cnt = 0
        for task in FMRI_TASKS:
            nii  = root/sub/"func"/f"{sub}_task-{task}_bold.nii.gz"
            vhdr = root/"derivatives"/sub/"eeg_pp"/f"{sub}_task-{task}_eeg_pp.vhdr"
            ev   = root/f"task-{task}_events.tsv"
            if not (nii.exists() and vhdr.exists() and ev.exists()): continue
            try:
                img = nib.load(str(nii)); data = np.asarray(img.dataobj, np.float32)
                if data.ndim != 4: continue
                ref = nib.Nifti1Image(data[...,0], img.affine, img.header)
                lab = np.asarray(resample_to_img(amap, ref, interpolation="nearest",
                        force_resample=True, copy_header=True).dataobj).astype(int)
                ts = np.zeros((data.shape[3], N_ROI), np.float32)
                for k in range(1, N_ROI+1):
                    m = lab == k
                    if m.sum() >= 10: ts[:,k-1] = data[m].mean(0)
                del data
                hm = root/"derivatives"/sub/"head_motion"/f"d_{sub}_task-{task}_headmotion.tsv"
                motion = None
                if hm.exists():
                    df = pd.read_csv(hm, sep="\t").select_dtypes("number").dropna(axis=1, how="all")
                    if df.shape[1] >= 6: motion = df.to_numpy()
                ts = _clean(ts, motion)[:, motor]

                raw = mne.io.read_raw_brainvision(str(vhdr), preload=True, verbose="ERROR")
                eeg = raw.get_data(picks=mne.pick_types(raw.info, eeg=True, exclude=[]))*1e6
                fs = float(raw.info["sfreq"])
                trig = np.array([o for o,dd in zip(raw.annotations.onset, raw.annotations.description)
                                 if "R128" in str(dd)])
                trig = (trig*fs).astype(int)
                if trig.size < EPOCH_VOL: continue

                e = pd.read_csv(ev, sep="\t")
                if "trial_type" in e.columns:
                    e = e[~e["trial_type"].astype(str).str.lower().str.contains("rest")]
                for o in pd.to_numeric(e["onset"], errors="coerce").dropna().to_numpy():
                    v = int(round(o/TR))
                    if v<0 or v+EPOCH_VOL>ts.shape[0] or v>=trig.size: continue
                    s0 = trig[v]
                    if s0+EEG_SAMPLES > eeg.shape[1]: continue
                    sh = ts[v:v+EPOCH_VOL].T; sh = sh - sh[:,:1]
                    se = eeg[:, s0:s0+EEG_SAMPLES]
                    se = (se-se.mean(-1,keepdims=True))/(se.std(-1,keepdims=True)+1e-6)
                    eeg_o.append(se.astype(np.float32)); hemo_o.append(sh.astype(np.float32))
                    sid_o.append(si); cnt += 1
                del eeg, raw
            except Exception as exc:
                print(f"  {sub}/{task}: {type(exc).__name__}: {str(exc)[:60]}")
        print(f"  {sub}: {cnt} trials")
    nc = min(x.shape[0] for x in eeg_o); nr = min(x.shape[0] for x in hemo_o)
    np.savez_compressed(out, eeg=np.stack([x[:nc] for x in eeg_o]),
                        hemo=np.stack([x[:nr] for x in hemo_o]),
                        subject_ids=np.asarray(sid_o))
    print(f"{out.name}: {len(eeg_o)} trials, {len(set(sid_o))} subjects")
    return out

if not cached("ds002336_fixed.npz"):
    root = fetch("ds002336")
    build_fmri(root)
    free("ds002336")
else:
    print("ds002336_fixed.npz: cached")

## 4. Load

Both arms are pooled into one array per dataset; the train/test division happens during
cross-validation, by subject.

In [ ]:
def load(name, modality, hemo_fs, chromophores):
    p = CACHE/name
    if not p.exists(): return None
    z = np.load(p, allow_pickle=True)
    return dict(eeg=z["eeg"], hemo=z["hemo"].astype(np.float64), sid=z["subject_ids"],
                modality=modality, hemo_fs=hemo_fs, chromophores=chromophores)

DATA = {}
for key, args in {"fNIRS hBCI": ("hbci_MI.npz","fnirs",10.0,True),
                  "fMRI ds002336": ("ds002336_fixed.npz","fmri",0.5,False)}.items():
    d = load(*args)
    if d is not None:
        DATA[key] = d
        print(f"{key:16s} {d['eeg'].shape[0]:5d} trials  {np.unique(d['sid']).size:2d} subjects"
              f"  eeg {d['eeg'].shape[1]}x{d['eeg'].shape[2]}  hemo {d['hemo'].shape[1]}x{d['hemo'].shape[2]}")
if not DATA: raise SystemExit("no caches built - check the download cells")

## 5. Target quality — the ceiling on everything

Split a subject's trials in half, average each, correlate. **A target that cannot predict
itself cannot be predicted by anything**, so this bounds every result below: the maximum
achievable correlation is about `sqrt(reliability)`.

This single check is what exposed the earlier failures — one dataset scored −0.639 while
still producing "significant" predictions.

In [ ]:
def concat_pearson(a, b):
    a, b = np.asarray(a).ravel(), np.asarray(b).ravel()
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 10 or a[ok].std() == 0 or b[ok].std() == 0: return np.nan
    return float(np.corrcoef(a[ok], b[ok])[0,1])

def subject_reliability(t, n_repeat=50, seed=0):
    rng = np.random.default_rng(seed); n = t.shape[0]
    if n < 6: return np.nan
    rs = [concat_pearson(t[p[:n//2]].mean(0), t[p[n//2:]].mean(0))
          for p in (rng.permutation(n) for _ in range(n_repeat))]
    rs = [r for r in rs if np.isfinite(r)]
    return float(np.mean(rs)) if rs else np.nan

def boot_ci(vals, n=N_BOOT, seed=0):
    v = np.asarray([x for x in vals if np.isfinite(x)])
    if v.size == 0: return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    b = [rng.choice(v, v.size, replace=True).mean() for _ in range(n)]
    return float(v.mean()), float(np.percentile(b,2.5)), float(np.percentile(b,97.5))

rows = []
for k, d in DATA.items():
    vals = [subject_reliability(d["hemo"][d["sid"]==s]) for s in np.unique(d["sid"])]
    m, lo, hi = boot_ci(vals)
    hh = np.nan
    if d["chromophores"]:
        nh = d["hemo"].shape[1]//2
        hh = float(np.nanmedian([np.corrcoef(d["hemo"][i,:nh].ravel(), d["hemo"][i,nh:].ravel())[0,1]
                                 for i in range(min(200, len(d["hemo"])))]))
    rows.append({"dataset": k, "modality": d["modality"],
                 "n_subjects": int(np.unique(d["sid"]).size),
                 "reliability": m, "ci_low": lo, "ci_high": hi,
                 "ceiling": float(np.sqrt(max(m,0))), "hbo_hbr_corr": hh})
qual = pd.DataFrame(rows)
print(qual.round(3).to_string(index=False))
print("\nHbO/HbR must be NEGATIVE for a real haemodynamic response.")
print("Reliability <= 0 means the target does not reproduce; any 'significant'")
print("correlation against such a target is spurious.")

## 6. Features, model, and the three metrics

`r_raw` alone is misleading: every trial shares a stereotyped HRF shape, so a predictor
that ignores EEG entirely still scores well. `r_demeaned` removes each subject's mean,
which drives the constant predictor to exactly 0 — what remains is the EEG contribution.

In [ ]:
def eeg_features(eeg, bands=BANDS, n_bins=N_BINS, erd=False):
    N, C, T = eeg.shape; L = T//n_bins
    x = eeg[:,:,:L*n_bins].reshape(N,C,n_bins,L)
    x = x - x.mean(-1, keepdims=True)
    P = np.abs(np.fft.rfft(x*np.hanning(L), axis=-1))**2
    f = np.fft.rfftfreq(L, 1/EEG_FS)
    bp = np.log10(np.stack([P[..., (f>=lo)&(f<hi)].sum(-1) for lo,hi in bands], -1)+1e-12)
    out = [bp.reshape(N,-1)]
    if erd: out.append((bp - bp[:,:,:1,:]).reshape(N,-1))
    return np.concatenate(out,1).astype(np.float32)

def ridge(Xtr, Ytr, Xte, alphas=(1e0,1e1,1e2,1e3,1e4,1e5)):
    mu, sd = Xtr.mean(0), Xtr.std(0)+1e-8
    Xtr, Xte = (Xtr-mu)/sd, (Xte-mu)/sd
    ym = Ytr.mean(0); Yc = Ytr-ym
    n = int(Xtr.shape[0]*0.8); best, best_r = alphas[len(alphas)//2], -np.inf
    if Xtr.shape[0]-n >= 4:
        G = Xtr[:n].T@Xtr[:n]
        for al in alphas:
            W = np.linalg.solve(G+al*np.eye(G.shape[0]), Xtr[:n].T@Yc[:n])
            r = concat_pearson(Xtr[n:]@W, Yc[n:])
            if np.isfinite(r) and r > best_r: best_r, best = r, al
    G = Xtr.T@Xtr
    return Xte @ np.linalg.solve(G+best*np.eye(G.shape[0]), Xtr.T@Yc) + ym

def demean(A, sid):
    out = A.copy()
    for s in np.unique(sid): out[sid==s] -= out[sid==s].mean(0, keepdims=True)
    return out

def perm_p(pred, true, n=N_PERM, seed=0):
    """Exact permutation null, computed fast.

    A permutation only REORDERS rows, so r(perm) = sum_i <pred[perm[i]], true[i]> / (|p||t|)
    with norms and global mean unchanged. All pairwise inner products are therefore computed
    ONCE as a matmul and each permutation becomes a gather - ~550x faster than
    re-correlating, and numerically identical.
    """
    P = np.asarray(pred, np.float32).reshape(len(pred), -1)
    T = np.asarray(true, np.float32).reshape(len(true), -1)
    Pc, Tc = P - P.mean(), T - T.mean()
    nP = np.sqrt((Pc.astype(np.float64)**2).sum()); nT = np.sqrt((Tc.astype(np.float64)**2).sum())
    if nP <= 0 or nT <= 0: return np.nan, np.nan, np.nan
    D = (Pc @ Tc.T).astype(np.float64); N = len(D); ar = np.arange(N)
    obs = float(D[ar,ar].sum()/(nP*nT))
    rng = np.random.default_rng(seed)
    null = np.array([D[rng.permutation(N), ar].sum() for _ in range(n)])/(nP*nT)
    return obs, float(null.mean()), float((np.sum(null>=obs)+1)/(null.size+1))

def folds_by_subject(sid, k=K_FOLDS, seed=0):
    subs = np.unique(sid); rng = np.random.default_rng(seed); rng.shuffle(subs)
    return [subs[i::k] for i in range(k)]

## 7. Cross-validated ridge

Subjects are split into 5 groups; each is held out once while the model trains on the rest.
No subject ever appears in both training and test. Reported as the mean across folds with a
bootstrap CI, plus a permutation p-value on the pooled out-of-fold predictions.

In [ ]:
def cv_ridge(d, channels=None, tag=""):
    X = eeg_features(d["eeg"]); H = d["hemo"]; sid = d["sid"]
    if channels is not None and d["chromophores"]:
        nh = H.shape[1]//2; H = H[:, np.concatenate([channels, channels+nh])]
    Y = H.reshape(len(H), -1)
    per, P, T, S = [], [], [], []
    for f, test_subs in enumerate(folds_by_subject(sid)):
        te = np.isin(sid, test_subs); tr = ~te
        if te.sum() < 6 or tr.sum() < 20: continue
        pred = ridge(X[tr], Y[tr], X[te])
        const = np.repeat(Y[tr].mean(0)[None,:], te.sum(), 0)
        per.append({"fold": f, "n_test": int(te.sum()),
                    "r_raw": concat_pearson(pred, Y[te]),
                    "r_const": concat_pearson(const, Y[te]),
                    "r_demeaned": concat_pearson(demean(pred, sid[te]), demean(Y[te], sid[te]))})
        P.append(pred); T.append(Y[te]); S.append(sid[te])
    fold_df = pd.DataFrame(per)
    Pm, Tm, Sm = np.vstack(P), np.vstack(T), np.concatenate(S)
    _, null, p = perm_p(demean(Pm, Sm), demean(Tm, Sm))
    m, lo, hi = boot_ci(fold_df.r_demeaned.values)
    gain = fold_df.r_raw.mean() - fold_df.r_const.mean()
    if tag:
        flag = "" if gain > 0 else "   <- constant predictor WINS on raw"
        print(f"{tag:16s} raw {fold_df.r_raw.mean():+.4f}  const {fold_df.r_const.mean():+.4f}"
              f"   DEMEANED {m:+.4f} [{lo:+.4f}, {hi:+.4f}]  p={p:.4f}{flag}")
    return {"r_raw_mean": fold_df.r_raw.mean(), "r_const_mean": fold_df.r_const.mean(),
            "r_gain_mean": gain, "r_demeaned_mean": m, "ci_low": lo, "ci_high": hi,
            "null": null, "p": p, "n_folds": len(fold_df)}, fold_df

cv_rows, cv_folds = [], {}
for k, d in DATA.items():
    s, fd = cv_ridge(d, tag=k)
    cv_rows.append({"dataset": k, "modality": d["modality"], **s}); cv_folds[k] = fd
cv = pd.DataFrame(cv_rows)
print("\nper-fold spread:")
for k, fd in cv_folds.items():
    print(f"  {k}:", np.round(fd.r_demeaned.values, 4))

## 8. Target-channel selection

The fNIRS target averages **all 36 channels**; many sit over non-responsive cortex and add
only noise. Ranking is recomputed **inside each fold** from training subjects only, so the
selection never sees held-out data.

In [ ]:
def channel_reliability(hemo, sid, n_repeat=20, seed=0):
    rng = np.random.default_rng(seed); n_ch = hemo.shape[1]
    acc, cnt = np.zeros(n_ch), np.zeros(n_ch)
    for s in np.unique(sid):
        idx = np.flatnonzero(sid == s)
        if idx.size < 6: continue
        for _ in range(n_repeat):
            p = rng.permutation(idx)
            a, b = hemo[p[:idx.size//2]].mean(0), hemo[p[idx.size//2:]].mean(0)
            for c in range(n_ch):
                if a[c].std() > 0 and b[c].std() > 0:
                    acc[c] += np.corrcoef(a[c], b[c])[0,1]; cnt[c] += 1
    return np.where(cnt > 0, acc/np.maximum(cnt,1), np.nan)

chan, BEST_K = pd.DataFrame(), None
d = DATA.get("fNIRS hBCI")
if d is not None:
    X, H, sid = eeg_features(d["eeg"]), d["hemo"], d["sid"]; nh = H.shape[1]//2
    rows_c = []
    for k in (4,8,12,18,24,36):
        per, P, T, S = [], [], [], []
        for test_subs in folds_by_subject(sid):
            te = np.isin(sid, test_subs); tr = ~te
            if te.sum() < 6 or tr.sum() < 20: continue
            rel = channel_reliability(H[tr], sid[tr])[:nh]          # TRAIN ONLY
            sel = np.sort(np.argsort(-np.nan_to_num(rel, nan=-9))[:k])
            Y = H[:, np.concatenate([sel, sel+nh])].reshape(len(H), -1)
            pred = ridge(X[tr], Y[tr], X[te])
            per.append(concat_pearson(demean(pred, sid[te]), demean(Y[te], sid[te])))
            P.append(pred); T.append(Y[te]); S.append(sid[te])
        _, null, p = perm_p(demean(np.vstack(P), np.concatenate(S)),
                            demean(np.vstack(T), np.concatenate(S)))
        m, lo, hi = boot_ci(per)
        rows_c.append({"k_channels": k, "r_demeaned": m, "ci_low": lo, "ci_high": hi, "p": p})
        print(f"  top-{k:<3d} channels  {m:+.4f} [{lo:+.4f}, {hi:+.4f}]  p={p:.4f}"
              f"{' *' if p<0.05 else ''}")
    chan = pd.DataFrame(rows_c)
    b = chan.loc[chan.r_demeaned.idxmax()]; all36 = chan[chan.k_channels==36].r_demeaned.iloc[0]
    BEST_K = int(b.k_channels)
    print(f"\nbest {BEST_K} channels -> {b.r_demeaned:+.4f}  (all 36 -> {all36:+.4f}, "
          f"change {b.r_demeaned-all36:+.4f})")

## 9. Deep encoder (EEGNet + TCN), cross-validated

Learning-rate search, augmentation (noise / channel dropout / time jitter), early stopping
on subjects held out **from the training folds**. Test subjects are never touched.

In [ ]:
import torch, torch.nn as nn
from scipy.signal import butter, filtfilt

def bandpower_seq(eeg, fs=EEG_FS, out_hz=20.0):
    N, C, T = eeg.shape; step = int(fs/out_hz); outs = []
    for lo, hi in BANDS:
        b, a = butter(4, [lo/(fs/2), min(hi/(fs/2), 0.99)], btype="band")
        env = filtfilt(b, a, eeg, axis=-1)**2
        outs.append(env[:,:,:(T//step)*step].reshape(N,C,T//step,step).mean(-1))
    x = np.log10(np.concatenate(outs,1)+1e-12).astype(np.float32)
    return (x - x.mean(-1,keepdims=True))/(x.std(-1,keepdims=True)+1e-6)

class TCN(nn.Module):
    def __init__(s, ch, dl):
        super().__init__()
        s.net = nn.Sequential(nn.Conv1d(ch,ch,3,padding=dl,dilation=dl), nn.BatchNorm1d(ch),
                              nn.ELU(), nn.Dropout(0.2),
                              nn.Conv1d(ch,ch,3,padding=dl,dilation=dl), nn.BatchNorm1d(ch), nn.ELU())
    def forward(s,x): return torch.relu(x + s.net(x))

class Enc(nn.Module):
    def __init__(s, ic, oc, ol, lat=64):
        super().__init__()
        s.sp = nn.Sequential(nn.Conv1d(ic,128,1), nn.BatchNorm1d(128), nn.ELU(), nn.Dropout(0.25),
                             nn.Conv1d(128,lat,1), nn.BatchNorm1d(lat), nn.ELU())
        s.t = nn.Sequential(*[TCN(lat,dd) for dd in (1,2,4,8)])
        s.p = nn.AdaptiveAvgPool1d(16)
        s.h = nn.Sequential(nn.Flatten(), nn.Linear(lat*16,512), nn.ELU(),
                            nn.Dropout(0.3), nn.Linear(512, oc*ol))
        s.oc, s.ol = oc, ol
    def forward(s,x): return s.h(s.p(s.t(s.sp(x)))).view(-1, s.oc, s.ol)

def augment(x, noise=0.05, ch_drop=0.1, jitter=3):
    x = x + torch.randn_like(x)*noise
    if ch_drop > 0:
        x = x * (torch.rand(x.shape[0], x.shape[1], 1, device=x.device) > ch_drop).float()
    if jitter > 0:
        x = torch.roll(x, shifts=int(torch.randint(-jitter, jitter+1, (1,)).item()), dims=-1)
    return x

def train_one(Xtr, Ytr, Xte, sid_tr):
    ys = Ytr.std()+1e-8
    subs = np.unique(sid_tr); n_in = max(1, int(len(subs)*0.8))
    m_tr = np.isin(sid_tr, subs[:n_in]); m_es = ~m_tr
    if m_es.sum() < 8: m_tr = np.ones(len(Xtr), bool); m_es = m_tr
    Xt = torch.tensor(Xtr[m_tr]); Yt = torch.tensor(Ytr[m_tr]/ys, dtype=torch.float32)
    Xe = torch.tensor(Xtr[m_es]).to(DEV); Ye = torch.tensor(Ytr[m_es]/ys, dtype=torch.float32).to(DEV)
    Xv = torch.tensor(Xte).to(DEV)
    best_all, best_state = np.inf, None
    for lr in LR_GRID:
        torch.manual_seed(0)
        m = Enc(Xtr.shape[1], Ytr.shape[1], Ytr.shape[2]).to(DEV)
        opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, MAX_EPOCHS)
        best, bad, state, n = np.inf, 0, None, len(Xt)
        for ep in range(MAX_EPOCHS):
            m.train(); perm = torch.randperm(n)
            for i in range(0, n, BS):
                idx = perm[i:i+BS]; opt.zero_grad()
                nn.functional.mse_loss(m(augment(Xt[idx].to(DEV))), Yt[idx].to(DEV)).backward()
                opt.step()
            sch.step(); m.eval()
            with torch.no_grad(): es = nn.functional.mse_loss(m(Xe), Ye).item()
            if es < best - 1e-5:
                best, bad = es, 0
                state = {k_: v.detach().clone() for k_, v in m.state_dict().items()}
            else:
                bad += 1
                if bad >= PATIENCE: break
        if best < best_all: best_all, best_state = best, state
    m = Enc(Xtr.shape[1], Ytr.shape[1], Ytr.shape[2]).to(DEV)
    m.load_state_dict(best_state); m.eval()
    with torch.no_grad():
        return torch.cat([m(Xv[i:i+64]).cpu() for i in range(0,len(Xv),64)]).numpy()*ys

deep_rows = []
for k, d in DATA.items():
    print(f"\n  {k}")
    X = bandpower_seq(d["eeg"]); H = d["hemo"]; sid = d["sid"]
    if k == "fNIRS hBCI" and BEST_K is not None:
        rel = channel_reliability(H, sid)[:H.shape[1]//2]
        sel = np.sort(np.argsort(-np.nan_to_num(rel, nan=-9))[:BEST_K])
        H = H[:, np.concatenate([sel, sel + H.shape[1]//2])]
    per, P, T, S = [], [], [], []
    for f, test_subs in enumerate(folds_by_subject(sid)):
        te = np.isin(sid, test_subs); tr = ~te
        if te.sum() < 6 or tr.sum() < 20: continue
        pred = train_one(X[tr], H[tr], X[te], sid[tr])
        pr = pred.reshape(len(pred),-1); tt = H[te].reshape(te.sum(),-1)
        per.append(concat_pearson(demean(pr, sid[te]), demean(tt, sid[te])))
        P.append(pr); T.append(tt); S.append(sid[te])
        print(f"    fold {f}: r_demeaned {per[-1]:+.4f}")
    _, null, p = perm_p(demean(np.vstack(P), np.concatenate(S)),
                        demean(np.vstack(T), np.concatenate(S)))
    m, lo, hi = boot_ci(per)
    deep_rows.append({"dataset": k, "modality": d["modality"],
                      "channels": "selected" if (k=="fNIRS hBCI" and BEST_K) else "all",
                      "r_demeaned_mean": m, "ci_low": lo, "ci_high": hi,
                      "null": null, "p": p, "n_folds": len(per)})
    print(f"  {k:16s} DEMEANED {m:+.4f} [{lo:+.4f}, {hi:+.4f}]  p={p:.4f}")
deep = pd.DataFrame(deep_rows)

## 10. Summary

In [ ]:
print("="*100)
print("TARGET QUALITY");          print(qual.round(3).to_string(index=False))
print("\nCROSS-VALIDATED RIDGE"); print(cv.round(4).to_string(index=False))
if len(chan): print("\nCHANNEL SELECTION"); print(chan.round(4).to_string(index=False))
print("\nCROSS-VALIDATED DEEP");  print(deep.round(4).to_string(index=False))
print("="*100)

q = qual.set_index("dataset"); r_ = cv.set_index("dataset"); dp_ = deep.set_index("dataset")
print(f"\n{'dataset':18s}{'reliab':>8s}{'ceiling':>9s}{'ridge':>10s}{'%ceil':>7s}"
      f"{'deep':>10s}{'%ceil':>7s}   significance")
for k in DATA:
    rel, ceil = q.loc[k,'reliability'], q.loc[k,'ceiling']
    rr, rd = r_.loc[k,'r_demeaned_mean'], dp_.loc[k,'r_demeaned_mean']
    f1 = f"{100*rr/ceil:.0f}%" if ceil>0 else "n/a"
    f2 = f"{100*rd/ceil:.0f}%" if ceil>0 else "n/a"
    print(f"{k:18s}{rel:>8.3f}{ceil:>9.3f}{rr:>10.4f}{f1:>7s}{rd:>10.4f}{f2:>7s}"
          f"   p_ridge={r_.loc[k,'p']:.4f} p_deep={dp_.loc[k,'p']:.4f}")

for name, df in [("target_quality",qual), ("cv_ridge",cv), ("cv_deep",deep), ("channels",chan)]:
    if len(df): df.to_csv(WORK/f"results_{name}.csv", index=False)
print(f"\nCSVs saved to {WORK}")
print("\nEvery figure is subject-wise cross-validated - each subject held out exactly once.")
print("Judge on r_demeaned: it is the only column where a predictor that ignores EEG")
print("scores zero, so it is the only one that measures the EEG contribution.")